### Combining Self-Measured Laptop Data with the Unified Dataset



 Imports

In [1]:
import pandas as pd
import numpy as np

Memory-string parser

In [2]:
def parse_mem(s):
    """Convert a memory string like '1.2 GB' to bytes. NaN if unparseable."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s == "" or s.lower() == "nan":
        return np.nan
    parts = s.split()
    if len(parts) < 2:                      # bare number / no unit -> can't interpret
        return np.nan
    try:
        num = float(parts[0])
    except ValueError:
        return np.nan
    factors = {"B": 1, "KB": 1024, "MB": 1048576, "GB": 1073741824}
    return num * factors.get(parts[1].upper(), np.nan)   # NaN if unit unrecognised

In [24]:
def parse_realtime_to_ms(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    total_ms = 0
    for part in s.split():
        if part.endswith("ms"):
            total_ms += float(part[:-2])
        elif part.endswith("s"):
            total_ms += float(part[:-1]) * 1000
        elif part.endswith("m"):
            total_ms += float(part[:-1]) * 60 * 1000
        elif part.endswith("h"):
            total_ms += float(part[:-1]) * 3600 * 1000
    return total_ms

Load the unified dataset (reference schema)

In [3]:
unified_dataset = pd.read_csv("../Processed_data/Unified_dataset/unified_dataset.csv")
print(unified_dataset.columns.tolist())

['Unnamed: 0', 'source_dataset', 'node', 'source_workflow', 'process', 'runtime_s', 'peak_mem', 'energy_j', 'rchar', 'cpus', 'concurrent_task_count', 'cores', 'ram', 'cpu_benchmark', 'io_read', 'io_write', 'task_type']


AMD Ryzen laptop — atacseq

In [91]:
amd_7_atac = pd.read_csv("../Processed_data/Experiments/AMD_Ryzen7/atacseq/measured/laptop_measured_energy.csv")
amd_7_atac = amd_7_atac.drop(columns=["Unnamed: 0"], errors="ignore")
amd_7_atac["source_workflow"]= "atacseq"
print("In unified but not AMD:", set(unified_dataset.columns) - set(amd_7_atac.columns))
print("In AMD but not unified:", set(amd_7_atac.columns) - set(unified_dataset.columns))

amd_7_atac["realtime"] = amd_7_atac["realtime"].apply(parse_realtime_to_ms)

amd_7_atac["peak_rss"]  = amd_7_atac["peak_rss"].apply(parse_mem)
amd_7_atac["source_workflow"]

In unified but not AMD: {'source_dataset', 'peak_mem', 'runtime_s', 'concurrent_task_count', 'cpus', 'node', 'ram', 'process', 'cpu_benchmark', 'io_read', 'energy_j', 'io_write', 'cores', 'Unnamed: 0'}
In AMD but not unified: {'name', 'complete_sec', 'complete_dt', 'exit', 'task_id', 'measured_energy_j', 'status', 'start', 'realtime', 'start_dt', '%cpu', 'peak_rss', 'start_sec', 'complete'}


0      atacseq
1      atacseq
2      atacseq
3      atacseq
4      atacseq
        ...   
260    atacseq
261    atacseq
262    atacseq
263    atacseq
264    atacseq
Name: source_workflow, Length: 265, dtype: str

In [92]:
amd_7_chip = pd.read_csv("../Processed_data/Experiments/AMD_Ryzen7/chipseq/measured/chipseq_laptop_measured.csv")
amd_7_chip = amd_7_chip.drop(columns=["Unnamed: 0"], errors="ignore")
amd_7_chip["source_workflow"]= "chipseq"
print("In unified but not AMD:", set(unified_dataset.columns) - set(amd_7_chip.columns))
print("In AMD but not unified:", set(amd_7_chip.columns) - set(unified_dataset.columns))
amd_7_chip["realtime"]

amd_7_chip["peak_rss"]

In unified but not AMD: {'source_dataset', 'peak_mem', 'runtime_s', 'concurrent_task_count', 'node', 'ram', 'cpu_benchmark', 'io_read', 'energy_j', 'io_write', 'cores', 'Unnamed: 0'}
In AMD but not unified: {'inv_ctxt', 'task_id', 'start', 'scratch', 'realtime', 'rss', 'memory', 'vol_ctxt', 'module', '%mem', 'disk', 'measured_energy_j', 'hostname', 'complete', 'native_id', 'name', 'time', 'submit', 'syscr', 'read_bytes', 'peak_vmem', 'syscw', 'tag', 'workdir', 'error_action', 'start_sec', 'queue', 'duration', 'complete_sec', 'exit', 'wchar', 'vmem', 'status', '%cpu', 'cpu_model', 'peak_rss', 'container', 'attempt'}


0        8937472
1        6553600
2       10493952
3      133464064
4        8790016
         ...    
210    112283648
211    331837440
212    332234752
213    173113344
214    813961216
Name: peak_rss, Length: 215, dtype: int64

In [93]:
amd = pd.concat([amd_7_atac,amd_7_chip])
print(amd["source_workflow"].value_counts())


source_workflow
atacseq    265
chipseq    215
Name: count, dtype: int64


In [173]:
amd = amd.rename(columns={"measured_energy_j": "energy_j", "peak_rss": "peak_mem", "realtime":"runtime_s"})
amd["source_dataset"]  = "amd_laptop"
amd["node"] = "amd_ryzen_7"

# fixed hardware spec for this machine
amd["cores"] = 8.0
amd["ram"]   = 16.0

print("AMD columns:", amd.columns.tolist())

amd["peak_mem"].dtype

AMD columns: ['task_id', 'name', 'status', 'exit', 'start', 'complete', 'runtime_s', '%cpu', 'peak_mem', 'rchar', 'start_dt', 'complete_dt', 'start_sec', 'complete_sec', 'task_type', 'energy_j', 'source_workflow', 'hostname', 'native_id', 'process', 'tag', 'module', 'container', 'cpus', 'time', 'disk', 'memory', 'attempt', 'submit', 'duration', 'queue', '%mem', 'rss', 'vmem', 'peak_vmem', 'wchar', 'syscr', 'syscw', 'read_bytes', 'vol_ctxt', 'inv_ctxt', 'workdir', 'scratch', 'error_action', 'cpu_model', 'source_dataset', 'node', 'cores', 'ram']


dtype('float64')

amd ryzen 5

In [154]:
amd_5_atac =  pd.read_csv("../Processed_data/Experiments/AMD_Ryzen5/atacseq/measured/atacseq_ryzen5_measured.csv")
amd_5_atac = amd_5_atac.drop(columns=["Unnamed: 0"], errors="ignore")

print("In unified but not AMD:", set(unified_dataset.columns) - set(amd_5_atac.columns))
print("In AMD but not unified:", set(amd_5_atac.columns) - set(unified_dataset.columns))

amd_5_atac["realtime"] = amd_5_atac["realtime"].apply(parse_realtime_to_ms)
amd_5_atac["source_workflow"] = "atacseq"
amd_5_atac["peak_rss"]  = amd_5_atac["peak_rss"].apply(parse_mem)
amd_5_atac["peak_mem"]


In unified but not AMD: {'concurrent_task_count', 'ram', 'cpu_benchmark', 'io_read', 'io_write', 'cores', 'Unnamed: 0'}
In AMD but not unified: {'inv_ctxt', 'task_id', 'start', 'scratch', 'realtime', 'rss', 'memory', 'vol_ctxt', 'module', 'write_bytes', '%mem', 'disk', 'measured_energy_j', 'hostname', 'complete', 'native_id', 'name', 'time', 'submit', 'syscr', 'read_bytes', 'peak_vmem', 'syscw', 'tag', 'workdir', 'error_action', 'start_sec', 'queue', 'duration', 'complete_sec', 'exit', 'wchar', 'vmem', 'status', '%cpu', 'cpu_model', 'peak_rss', 'container', 'attempt'}


0        8650752
1        9961472
2      132538368
3        6422528
4        7471104
         ...    
260     10223616
261    710361088
262      2883584
263      9306112
264    237584384
Name: peak_mem, Length: 265, dtype: int64

In [155]:
amd_5_chip = pd.read_csv("../Processed_data/Experiments/AMD_Ryzen5/chipseq/measured/chipseq_ryzen5_measured.csv")
amd_5_chip = amd_5_chip.drop(columns=["Unnamed: 0"], errors="ignore")
print("In unified but not AMD:", set(unified_dataset.columns) - set(amd_5_chip.columns))
print("In AMD but not unified:", set(amd_5_chip.columns) - set(unified_dataset.columns))
amd_5_chip["realtime"]
amd_5_chip["source_workflow"] = "chipseq"
amd_5_chip["peak_rss"]= amd_5_chip["peak_rss"].apply(parse_mem)


In unified but not AMD: {'concurrent_task_count', 'ram', 'cpu_benchmark', 'io_read', 'io_write', 'cores', 'Unnamed: 0'}
In AMD but not unified: {'inv_ctxt', 'task_id', 'start', 'scratch', 'realtime', 'rss', 'memory', 'vol_ctxt', 'module', 'write_bytes', '%mem', 'disk', 'measured_energy_j', 'hostname', 'complete', 'native_id', 'name', 'time', 'submit', 'syscr', 'read_bytes', 'peak_vmem', 'syscw', 'tag', 'workdir', 'error_action', 'start_sec', 'queue', 'duration', 'complete_sec', 'exit', 'wchar', 'vmem', 'status', '%cpu', 'cpu_model', 'peak_rss', 'container', 'attempt'}


In [156]:
amd_5 = pd.concat([amd_5_atac,amd_5_chip])
amd_5["energy_j"]

0       45.12
1       28.87
2      173.43
3      121.47
4       38.63
        ...  
205     71.31
206    134.51
207     69.73
208     23.97
209    804.64
Name: energy_j, Length: 475, dtype: float64

In [170]:

amd_5["source_dataset"]  = "amd_laptop"
amd_5["node"] = "amd_ryzen_5"

# fixed hardware spec for this machine
amd_5["cores"] = 6.0
amd_5["ram"]   = 16.0

print("AMD columns:", amd_5.columns.tolist())

amd_5["peak_mem"].dtype


AMD columns: ['task_id', 'hostname', 'native_id', 'process', 'tag', 'name', 'status', 'exit', 'module', 'container', 'cpus', 'time', 'disk', 'memory', 'attempt', 'submit', 'start', 'complete', 'duration', 'realtime', 'queue', '%cpu', '%mem', 'rss', 'vmem', 'peak_rss', 'peak_vmem', 'rchar', 'wchar', 'syscr', 'syscw', 'read_bytes', 'write_bytes', 'vol_ctxt', 'inv_ctxt', 'workdir', 'scratch', 'error_action', 'cpu_model', 'start_sec', 'complete_sec', 'task_type', 'measured_energy_j', 'source_dataset', 'source_workflow', 'node', 'runtime_s', 'peak_mem', 'energy_j', 'cores', 'ram']


dtype('int64')

Intel i5 laptop — chipseq

In [158]:
intel_atac = pd.read_csv("../Processed_data/Experiments/intel_i5/atacseq/measured/intel_atacseq_measured.csv")
intel_atac = intel_atac.drop(columns=["Unnamed: 0"], errors="ignore")
intel_atac = intel_atac.rename(columns={"measured_energy_j": "energy_j", "peak_rss": "peak_mem", "realtime" : "runtime_s"})
intel_atac["source_dataset"]  = "intel_i5_laptop"
intel_atac["source_workflow"] = "atacseq"
# runtime ms -> s

intel_atac["cores"] = 6.0
intel_atac["ram"]   = 32.0

if intel_atac["peak_mem"].dtype == object:
    intel_atac["peak_mem"] = intel_atac["peak_mem"].apply(parse_mem)


In [159]:
intel_chipseq = pd.read_csv("../Processed_data/Experiments/intel_i5/chipseq/measured/intel_chipseq_measured.csv")
intel_chipseq = intel_chipseq.drop(columns=["Unnamed: 0"], errors="ignore")
intel_chipseq = intel_chipseq.rename(columns={"measured_energy_j": "energy_j", "peak_rss": "peak_mem", "realtime" : "runtime_s"})
intel_chipseq["source_dataset"]  = "intel_i5_laptop"
intel_chipseq["source_workflow"] = "chipseq"


intel_chipseq["cores"] = 6.0
intel_chipseq["ram"]   = 32.0

if intel_chipseq["peak_mem"].dtype == object:
    intel_chipseq["peak_mem"] = intel_chipseq["peak_mem"].apply(parse_mem)

In [171]:
intel = pd.concat([intel_atac,intel_chipseq])
intel["peak_mem"]

0        3661824
1        3444736
2        4718592
3        6373376
4        8974336
         ...    
171    294162432
172    221458432
173     66326528
174    237203456
175    221999104
Name: peak_mem, Length: 371, dtype: int64

### Sanity check: units after conversion

In [161]:
for col in ["peak_mem", "runtime_s", "energy_j"]:
    print(f"{col}:"); print(intel[col].describe(), "\n")

peak_mem:
count    3.710000e+02
mean     2.112556e+08
std      4.736573e+08
min      0.000000e+00
25%      7.839744e+06
50%      1.968128e+07
75%      2.170020e+08
max      2.724246e+09
Name: peak_mem, dtype: float64 

runtime_s:
count      371.000000
mean      4419.045822
std      11252.743204
min          0.000000
25%          0.000000
50%       1000.000000
75%       3000.000000
max      64000.000000
Name: runtime_s, dtype: float64 

energy_j:
count     371.000000
mean      312.818051
std       757.641690
min         5.434190
25%        47.858703
50%       107.571197
75%       243.841723
max      4533.780874
Name: energy_j, dtype: float64 



Cross-source validation

In [162]:
print("Median peak_mem (bytes)")
print("  AMD 7   :", f"{amd['peak_mem'].median():,.0f}")
print("  AMD 5   :", f"{amd_5['peak_mem'].median():,.0f}")

print("  Intel  :", f"{intel['peak_mem'].median():,.0f}")
print("  Unified:", f"{unified_dataset['peak_mem'].median():,.0f}")

print("\nMissing from AMD  :", set(unified_dataset.columns) - set(amd.columns))
print("Missing from Intel:", set(unified_dataset.columns) - set(intel.columns))

Median peak_mem (bytes)
  AMD 7   : 14,837,350
  AMD 5   : 13,824,000
  Intel  : 19,681,280
  Unified: 286,609,408

Missing from AMD  : {'concurrent_task_count', 'cpu_benchmark', 'io_read', 'io_write', 'Unnamed: 0'}
Missing from Intel: {'concurrent_task_count', 'cpu_benchmark', 'io_read', 'io_write', 'Unnamed: 0'}


Combine all sources

In [166]:
cols = unified_dataset.columns.tolist()
amd_7_aligned   = amd[[c for c in cols if c in amd.columns]]
amd_5_aligned = amd_5[[c for c in cols if c in amd_5.columns]]
intel_aligned = intel[[c for c in cols if c in intel.columns]]

combined = pd.concat([unified_dataset, amd_7_aligned, amd_5_aligned, intel_aligned], ignore_index=True)
print(f"Combined shape: {combined.shape}")
print(combined["source_dataset"].value_counts())

Combined shape: (23998, 17)
source_dataset
augur_hu           11855
lotaru              8824
augur_gu            1993
amd_laptop           955
intel_i5_laptop      371
Name: count, dtype: int64


In [175]:

combined["rchar"] = combined["rchar"].apply(parse_mem)

### Save the combined dataset

In [178]:
combined.to_csv("../Processed_data/Augmented_combined_dataset/new_combined_dataset.csv", index=False)


In [179]:
feat_cols = ["rchar", "cpus", "concurrent_task_count"]
for col in feat_cols + ["energy_j", "runtime_s", "peak_mem"]:
    if col in combined.columns:
        strings = combined[col].apply(lambda x: isinstance(x, str)).sum()
        print(f"{col}: dtype={combined[col].dtype}, string_values={strings}")

rchar: dtype=float64, string_values=0
cpus: dtype=float64, string_values=0
concurrent_task_count: dtype=float64, string_values=0
energy_j: dtype=float64, string_values=0
runtime_s: dtype=float64, string_values=0
peak_mem: dtype=float64, string_values=0
